# SentinelAI — Notebook 05: MedDRA Hybrid Search

**Goal:** Given a free-text MAUDE adverse event narrative, find the most relevant MedDRA Preferred Terms (PTs) using a hybrid retrieval approach.

**Why hybrid?**  
A single retrieval method is not enough for clinical text:
- **BM25 alone** fails when the narrative uses synonyms or abbreviations not in the PT name (e.g. *"low blood sugar"* vs. *"Hypoglycaemia"*)
- **Vector search alone** can miss exact matches when terminology is consistent, and may hallucinate semantically related but wrong PTs
- **Hybrid = best of both worlds**: lexical precision + semantic recall

**Architecture of this notebook:**
1. Explain the two retrieval arms (BM25 via pg_trgm, Vector via pgvector)
2. Explain RRF fusion
3. Run live queries against the DB
4. Analyse result quality on example MAUDE narratives
5. Visualise rank contributions from each arm

---
## 1. The Two Retrieval Arms

### Arm 1: BM25 (Lexical) via pg_trgm

**BM25** (Best Match 25) is the standard ranking function for keyword search — it is the algorithm behind Elasticsearch and most search engines.  
In SentinelAI we approximate BM25 using **PostgreSQL pg_trgm** (trigram similarity):

- A **trigram** is a sequence of 3 characters: e.g. *"hypo"* → `{" hy", "hyp", "ypo", "po "}`
- pg_trgm computes the Jaccard similarity between the trigram sets of two strings
- This handles partial matches, abbreviations, and minor spelling variations

We search both `pt_name` (the standard PT) and `llt_name` (synonyms/variants) to catch cases where the narrative uses a non-preferred term.

**Example:**  
Query: *"hyperglycemia"*  
- `pt_name` match: *"Hyperglycaemia"* (British spelling) → trigram similarity ~0.7
- `llt_name` match: *"Hyperglycemia"* (American spelling) → trigram similarity ~1.0 → maps to same PT

### Arm 2: Vector Search via pgvector

Each MedDRA PT name was encoded into a **768-dimensional vector** using **PubMedBERT** (`microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract`) — a BERT model pre-trained on 21M PubMed abstracts.  
It understands clinical language far better than general-purpose BERT.

At query time:
1. The input text is encoded by the same PubMedBERT model → query vector
2. PostgreSQL's **pgvector** extension computes cosine similarity between the query vector and all 27k PT embeddings
3. The **IVFFlat index** (Inverted File with Flat quantization) makes this fast via Approximate Nearest Neighbour (ANN) search

**Why cosine similarity?**  
Cosine similarity measures the *angle* between two vectors, not their magnitude. This makes it robust to text length differences — a short PT name and a long narrative can still match well if they describe the same concept.

> **Cosine similarity = 1 − cosine distance**  
> pgvector's `<=>` operator returns cosine **distance**, so we compute `1 - (embedding <=> query)`

---
## 2. Reciprocal Rank Fusion (RRF)

**RRF** is a simple but highly effective method for combining multiple ranked lists into a single ranking.

For each document $d$ retrieved by at least one arm:

$$\text{score}_{RRF}(d) = \sum_{i \in \{\text{bm25, vector}\}} \frac{w_i}{k + \text{rank}_i(d)}$$

Where:
- $\text{rank}_i(d)$ = position of document $d$ in arm $i$'s ranked list (1 = best)
- $k = 60$ = smoothing constant that reduces the impact of rank gaps at the top
- $w_{bm25} = 0.4$, $w_{vector} = 0.6$ = arm weights

**Why $k = 60$?**  
Without $k$, rank 1 would score infinitely better than rank 2. $k=60$ is the empirically validated default from the original RRF paper (Cormack et al., 2009).

**Why $w_{vector} > w_{bm25}$?**  
MAUDE narratives use consumer/non-standard language. The semantic arm bridges vocabulary gaps better than trigram matching for clinical adverse event text.

**Key property of RRF:** It only uses *ranks*, not raw scores. This makes it robust — you don't need to calibrate trigram similarity (0.0–1.0) against cosine similarity (0.0–1.0) on the same scale.

In [ ]:
import sys
sys.path.insert(0, '..')  # allow imports from src/

import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from dotenv import load_dotenv
import psycopg2

load_dotenv()

# DB connection -- reads DATABASE_URL or individual vars from .env
def get_db_url():
    url = os.getenv('DATABASE_URL')
    if url:
        return url
    host = os.getenv('POSTGRES_HOST', 'localhost')
    port = os.getenv('POSTGRES_PORT', '5432')
    db   = os.getenv('POSTGRES_DB', 'vigilex')
    user = os.getenv('POSTGRES_USER', 'vigilex')
    pw   = os.getenv('POSTGRES_PASSWORD', '')
    return f'postgresql://{user}:{pw}@{host}:{port}/{db}'

conn = psycopg2.connect(get_db_url())
print('DB connected.')

---
## 3. Load the Hybrid Searcher

In [ ]:
from src.vigilex.coding.hybrid_search import HybridSearcher, EmbeddingModel

# Load PubMedBERT once -- reused across all queries
print('Loading PubMedBERT (first time: ~440MB download, then cached)...')
model = EmbeddingModel()
print(f'Model loaded on: {model.device}')

searcher = HybridSearcher(conn, embedding_model=model, candidate_pool=100)
print('HybridSearcher ready.')

---
## 4. Example Queries — MAUDE Narrative Excerpts

We test with realistic MAUDE narrative fragments covering different scenarios:
- Clear clinical terminology (should be easy)
- Consumer/layperson language (harder for BM25, semantic arm helps)
- Device-specific language (unique to medical device reports)
- Ambiguous / multi-symptom narratives

In [ ]:
TEST_QUERIES = [
    # (label, query text)
    ("Clear clinical term",
     "Patient experienced hypoglycaemia after insulin bolus delivery"),

    ("Consumer language",
     "blood sugar dropped very low, patient was shaking and confused"),

    ("Device malfunction",
     "insulin pump stopped delivering insulin, occlusion alarm triggered"),

    ("Skin reaction",
     "redness and swelling at the infusion site, possible allergic reaction"),

    ("CGM inaccuracy",
     "continuous glucose monitor showed incorrect readings, sensor failure"),

    ("Multi-symptom",
     "patient reported nausea, dizziness and loss of consciousness"),
]

print(f'{len(TEST_QUERIES)} test queries defined.')

In [ ]:
def results_to_df(results, label=''):
    rows = []
    for r in results:
        rows.append({
            'query_label':  label,
            'pt_code':      r.pt_code,
            'pt_name':      r.pt_name,
            'soc_name':     r.soc_name,
            'rrf_score':    round(r.rrf_score, 5),
            'bm25_rank':    r.bm25_rank,
            'vector_rank':  r.vector_rank,
            'trgm_sim':     round(r.trgm_sim, 3) if r.trgm_sim else None,
            'cosine_sim':   round(r.cosine_sim, 3) if r.cosine_sim else None,
        })
    return pd.DataFrame(rows)

In [ ]:
# Run all test queries
all_results = {}

for label, query in TEST_QUERIES:
    print(f'\n=== {label} ===')
    print(f'Query: "{query}"')
    results = searcher.search(query, top_k=5)
    all_results[label] = results

    df = results_to_df(results, label)
    print(df[['pt_name', 'soc_name', 'rrf_score', 'bm25_rank', 'vector_rank']].to_string(index=False))

---
## 5. Deep Dive: BM25 vs Vector vs Hybrid

Here we compare the three approaches for one query side-by-side to see what each arm contributes.

In [ ]:
# Pick the "Consumer language" query for comparison
QUERY = "blood sugar dropped very low, patient was shaking and confused"

# Get hybrid results (already computed above)
hybrid_results = all_results["Consumer language"]
df_hybrid = results_to_df(hybrid_results, 'Hybrid')

# BM25-only: sort by trigram similarity
df_bm25_only = df_hybrid.dropna(subset=['trgm_sim']).sort_values('trgm_sim', ascending=False).head(5).copy()
df_bm25_only['method'] = 'BM25 only'

# Vector-only: sort by cosine similarity
df_vec_only = df_hybrid.dropna(subset=['cosine_sim']).sort_values('cosine_sim', ascending=False).head(5).copy()
df_vec_only['method'] = 'Vector only'

df_hybrid_show = df_hybrid.head(5).copy()
df_hybrid_show['method'] = 'Hybrid (RRF)'

print(f'Query: "{QUERY}"\n')
print('--- BM25 only (trigram similarity) ---')
print(df_bm25_only[['pt_name', 'trgm_sim']].to_string(index=False))
print('\n--- Vector only (cosine similarity) ---')
print(df_vec_only[['pt_name', 'cosine_sim']].to_string(index=False))
print('\n--- Hybrid RRF (final ranking) ---')
print(df_hybrid_show[['pt_name', 'rrf_score', 'bm25_rank', 'vector_rank']].to_string(index=False))

---
## 6. Rank Contribution Visualisation

For each result in the hybrid output, we visualise how much each arm contributed to the final RRF score.

In [ ]:
from src.vigilex.coding.hybrid_search import RRF_K, RRF_W_BM25, RRF_W_VECTOR

def plot_rank_contributions(results, title='Hybrid Search — Rank Contributions'):
    labels = [r.pt_name[:40] for r in results]
    bm25_contribs   = [RRF_W_BM25   / (RRF_K + r.bm25_rank)   if r.bm25_rank   else 0 for r in results]
    vector_contribs = [RRF_W_VECTOR / (RRF_K + r.vector_rank) if r.vector_rank else 0 for r in results]

    x = range(len(labels))
    fig, ax = plt.subplots(figsize=(12, 5))

    bars1 = ax.bar(x, bm25_contribs,   label='BM25 (lexical)',   color='steelblue', alpha=0.85)
    bars2 = ax.bar(x, vector_contribs, label='Vector (semantic)', color='coral',     alpha=0.85,
                   bottom=bm25_contribs)

    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('RRF score contribution')
    ax.set_title(title, fontsize=12)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_rank_contributions(
    all_results["Consumer language"],
    title='Consumer language query — RRF rank contributions per PT'
)

In [ ]:
# Repeat for all queries
for label, results in all_results.items():
    plot_rank_contributions(results, title=f'{label} — RRF rank contributions')

---
## 7. Performance Benchmark

How fast is a single hybrid search query? This matters for the coding worker throughput.

In [ ]:
import time

BENCHMARK_QUERY = "patient experienced hypoglycaemia after insulin bolus delivery"
N_RUNS = 10

times = []
for i in range(N_RUNS):
    t0 = time.time()
    searcher.search(BENCHMARK_QUERY, top_k=10)
    times.append(time.time() - t0)

print(f'Benchmark over {N_RUNS} runs:')
print(f'  Mean:   {sum(times)/len(times)*1000:.1f} ms')
print(f'  Min:    {min(times)*1000:.1f} ms')
print(f'  Max:    {max(times)*1000:.1f} ms')
print()
print('Note: includes PubMedBERT query encoding + 2 DB queries + RRF fusion.')
print('Encoding dominates (~80% of time). For batch coding, encode all queries first, then query DB.')

---
## 8. Summary & Next Steps

**What we built:**
- `hybrid_search.py` — a `HybridSearcher` class that takes a free-text query and returns ranked MedDRA PTs
- Two retrieval arms: BM25 (pg_trgm) + Vector (pgvector + PubMedBERT)
- RRF fusion with weights tuned for clinical adverse event text

**What this is NOT yet:**
- Not a full MedDRA *coder* — hybrid search gives us Top-K candidates, but the final PT selection needs a reranker + LLM confidence score
- Not integrated with the MAUDE ingestion pipeline yet

**Next steps (Block D remaining):**
1. **CrossEncoder reranker** — MiniLM-L-6 scores each (query, PT) pair more precisely than the bi-encoder embedding
2. **LLM confidence** — Ollama (llama3.2) assigns a final confidence score and selects the single best PT
3. **Coding worker** — wraps hybrid search + reranker + LLM into a worker that processes MAUDE reports from the queue